# HaLRTC ????????

?? `data` ?????????????? `results/HaLRTC`????? MSE?RMSE?RSE?MAE ??????

In [1]:

import numpy as np
import pandas as pd
import time
from pathlib import Path


def ten2mat(tensor, mode):
    return np.reshape(np.moveaxis(tensor, mode, 0), (tensor.shape[mode], -1), order='F')


def mat2ten(mat, tensor_size, mode):
    index = [mode]
    for i in range(tensor_size.shape[0]):
        if i != mode:
            index.append(i)
    return np.moveaxis(np.reshape(mat, list(tensor_size[index]), order='F'), 0, mode)


def svt(mat, lambda0):
    u, s, v = np.linalg.svd(mat, full_matrices=False)
    vec = s - lambda0
    vec[vec < 0] = 0
    return np.matmul(np.matmul(u, np.diag(vec)), v)


def tensor_metrics(Xtrue, Xhat):
    diff = Xtrue.astype(float) - Xhat.astype(float)
    mse = float(np.mean(diff ** 2))
    rmse = float(np.sqrt(mse))
    rse = float(np.linalg.norm(diff.ravel()) / max(np.linalg.norm(Xtrue.ravel()), np.finfo(float).eps))
    mae = float(np.mean(np.abs(diff)))
    return mse, rmse, rse, mae


In [2]:

def HaLRTC(Xtrue, Omega, alpha, rho, maxiter, epsilon=1e-4, verbose=True):
    Omega = Omega.astype(bool)
    dim0 = Xtrue.ndim
    dim1, dim2, dim3 = Xtrue.shape
    sparse_tensor = Xtrue.astype(float) * Omega
    tensor_hat = sparse_tensor.copy()
    position = np.where(Omega)
    Z = np.zeros((dim1, dim2, dim3, dim0))
    T = np.zeros((dim1, dim2, dim3, dim0))
    last_tensor = tensor_hat.copy()
    train_norm = max(np.linalg.norm(sparse_tensor.ravel()), np.finfo(float).eps)
    start_time = time.perf_counter()
    rows = []

    for it in range(maxiter):
        for k in range(dim0):
            Z[:, :, :, k] = mat2ten(
                svt(ten2mat(tensor_hat + T[:, :, :, k] / rho, k), alpha / rho),
                np.array([dim1, dim2, dim3]),
                k,
            )

        tensor_hat = np.mean(Z - T / rho, axis=3)
        tensor_hat[position] = sparse_tensor[position]

        for k in range(dim0):
            T[:, :, :, k] = T[:, :, :, k] + rho * (tensor_hat - Z[:, :, :, k])

        tensor_recovery = np.clip(np.nan_to_num(tensor_hat, nan=0.0, posinf=1.0, neginf=0.0), 0, 1)
        mse, rmse, rse, mae = tensor_metrics(Xtrue, tensor_recovery)
        relative_change = float(np.linalg.norm((tensor_hat - last_tensor).ravel()) / train_norm)
        last_tensor = tensor_hat.copy()
        rows.append({
            'iteration': it + 1,
            'elapsed_time_seconds': time.perf_counter() - start_time,
            'MSE': mse,
            'RMSE': rmse,
            'RSE': rse,
            'MAE': mae,
            'relative_change': relative_change,
        })
        if verbose:
            print(f'Epoch = {it + 1}, MSE = {mse:.10f}, RMSE = {rmse:.10f}')
        if relative_change < epsilon:
            break

    history = pd.DataFrame(rows)
    tensor_recovery = np.clip(np.nan_to_num(tensor_hat, nan=0.0, posinf=1.0, neginf=0.0), 0, 1)
    return tensor_recovery, history, time.perf_counter() - start_time


In [3]:

base_dir = Path.cwd()
data_dir = base_dir / 'data'
result_root = base_dir / 'results' / 'HaLRTC'
result_root.mkdir(parents=True, exist_ok=True)

seed_list = [920, 921, 922]
miss_list = [0.8, 0.9, 0.95]
alpha_list = [1e-4, 1e-3, 1e-2, 1e-1, 1]
rho = 0.01
maxiter = 1000
epsilon = 1e-4
all_summary = []


def enrich_history(history, data_name, seed, missing_rate, alpha, status):
    history = history.copy()
    history.insert(0, 'dataset', data_name)
    history.insert(1, 'method', 'HaLRTC')
    history.insert(2, 'variant', 'default')
    history.insert(3, 'seed', seed)
    history.insert(4, 'missing_rate', missing_rate)
    history['parameter_settings'] = f'alpha={alpha}, rho={rho}, maxiter={maxiter}, epsilon={epsilon}'
    history['convergence_status'] = status
    return history


for seed in seed_list:
    for missing_rate in miss_list:
        miss_tag = int(round(missing_rate * 100))
        data_name = f'S{seed}_miss{miss_tag}'
        data = np.load(data_dir / f'{data_name}.npz', allow_pickle=True)
        Xtrue = data['X'].astype(float)
        Omega = data['Omega'].astype(bool)
        Y = data['Y'].astype(float)
        result_dir = result_root / data_name
        result_dir.mkdir(parents=True, exist_ok=True)

        print('\n' + '=' * 60)
        print('Method: HaLRTC')
        print(f'Data: {data_name}')
        print(f'Size: {Xtrue.shape}')
        print(f'Missing rate: {missing_rate:.2f}')
        print(f'Observed entries: {int(Omega.sum())} / {Omega.size}')
        print('=' * 60)

        best_mse = np.inf
        best = None
        summary_rows = []
        for alpha in alpha_list:
            status = 'ok'
            error_msg = ''
            try:
                Xhat, history, elapsed = HaLRTC(Xtrue, Omega, alpha, rho, maxiter, epsilon, verbose=False)
                Xhat = np.clip(Xhat, 0, 1)
                mse, rmse, rse, mae = tensor_metrics(Xtrue, Xhat)
                rel = float(history['relative_change'].iloc[-1]) if len(history) else np.nan
            except Exception as exc:
                status = 'error'
                error_msg = repr(exc)
                Xhat = Y.copy()
                history = pd.DataFrame(columns=['iteration', 'elapsed_time_seconds', 'MSE', 'RMSE', 'RSE', 'MAE', 'relative_change'])
                elapsed = 0.0
                mse = rmse = rse = mae = rel = np.nan
                print(f'alpha={alpha} failed: {error_msg}')

            history = enrich_history(history, data_name, seed, missing_rate, alpha, status)
            row = {
                'Data': data_name,
                'Method': 'HaLRTC',
                'Variant': 'default',
                'MissingRate': missing_rate,
                'Seed': seed,
                'alpha': alpha,
                'rho': rho,
                'maxiter': maxiter,
                'epsilon': epsilon,
                'MSE': mse,
                'RMSE': rmse,
                'RSE': rse,
                'MAE': mae,
                'RelativeChange': rel,
                'Time': elapsed,
                'Status': status,
                'Error': error_msg,
            }
            summary_rows.append(row)
            if status == 'ok' and np.isfinite(mse) and mse < best_mse:
                best_mse = mse
                best = {'alpha': alpha, 'Xhat': Xhat, 'history': history, **row}
            print(f'alpha={alpha}, MSE={mse:.10f}, RMSE={rmse:.10f}, status={status}')

        if best is None:
            best = {'alpha': np.nan, 'Xhat': Y.copy(), 'history': enrich_history(pd.DataFrame(columns=['iteration', 'elapsed_time_seconds', 'MSE', 'RMSE', 'RSE', 'MAE', 'relative_change']), data_name, seed, missing_rate, np.nan, 'all_failed'), **summary_rows[-1]}

        summary = pd.DataFrame(summary_rows)
        summary.to_csv(result_dir / 'summary.csv', index=False, encoding='utf-8-sig')
        summary.to_excel(result_dir / '\u5b9e\u9a8c\u603b\u7ed3.xlsx', index=False)
        best['history'].to_csv(result_dir / '\u6700\u4f73\u8fed\u4ee3.csv', index=False, encoding='utf-8-sig')
        best['history'].to_csv(result_dir / 'best_iteration_history.csv', index=False, encoding='utf-8-sig')
        np.savez_compressed(result_dir / 'result.npz', data_name=data_name, X=Xtrue, Omega=Omega, Y=Y, recovered=best['Xhat'], alpha=best['alpha'], rho=rho, maxiter=maxiter, epsilon=epsilon, missing_rate=missing_rate, seed=seed)
        with open(result_dir / 'metrics.txt', 'w', encoding='utf-8') as f:
            f.write(f'data={data_name}\nmethod=HaLRTC\nvariant=default\nseed={seed}\nmissing_rate={missing_rate:.2f}\n')
            f.write(f'alpha={best["alpha"]}\nrho={rho}\nmaxiter={maxiter}\nepsilon={epsilon}\n')
            f.write(f'MSE={best["MSE"]:.10f}\nRMSE={best["RMSE"]:.10f}\nRSE={best["RSE"]:.10f}\nMAE={best["MAE"]:.10f}\nrelative_change={best["RelativeChange"]:.10f}\ntime={best["Time"]:.6f}\nstatus={best["Status"]}\nerror={best["Error"]}\n')
        all_summary.append({k: best[k] for k in ['Data','Method','Variant','MissingRate','Seed','alpha','rho','maxiter','epsilon','MSE','RMSE','RSE','MAE','RelativeChange','Time','Status']})

all_table = pd.DataFrame(all_summary)
all_table.to_csv(result_root / 'all_summary.csv', index=False, encoding='utf-8-sig')
all_table.to_excel(result_root / '\u5168\u90e8\u5b9e\u9a8c\u603b\u7ed3.xlsx', index=False)
print('All HaLRTC low rank tensor experiments finished.')



Method: HaLRTC
Data: S920_miss80
Size: (200, 200, 30)
Missing rate: 0.80
Observed entries: 239385 / 1200000


alpha=0.0001, MSE=0.2025007818, RMSE=0.4500008686, status=ok


alpha=0.001, MSE=0.0203353204, RMSE=0.1426019650, status=ok


alpha=0.01, MSE=0.0000095611, RMSE=0.0030920974, status=ok


alpha=0.1, MSE=0.0000093010, RMSE=0.0030497510, status=ok


...

alpha=0.001, MSE=0.1937051463, RMSE=0.4401194682, status=ok


alpha=0.01, MSE=0.0002578392, RMSE=0.0160573709, status=ok


alpha=0.1, MSE=0.0002394668, RMSE=0.0154747148, status=ok


alpha=1, MSE=0.3329789136, RMSE=0.5770432510, status=ok


All HaLRTC low rank tensor experiments finished.
